In [ ]:
!pip install streamlit streamlit-folium localtunnel

ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import MarkerCluster
from streamlit_folium import folium_static

# Load the data (Pastikan file CSV sudah diunggah ke folder Colab Anda)
data_1 = pd.read_csv('/content/hasil_cluster_rf.csv')
data_2 = pd.read_csv('/content/perbandingan_cluster_periode.csv')

# Select period
period = st.sidebar.selectbox("Select Period", ["Pra-COVID", "Saat-COVID", "Pasca-COVID"])

# Initialize tahun_filter_value and tahun_display for clear handling
tahun_filter_value = None
tahun_display = ""

if period in ["Saat-COVID", "Pasca-COVID"]:
    available_years = sorted(data_1['tahun'].unique())
    tahun_filter_value = st.sidebar.selectbox("Pilih Tahun", available_years, key=f"tahun_selector_{period}")
    base_data_from_1 = data_1[data_1['tahun'] == tahun_filter_value].copy()
    tahun_display = str(tahun_filter_value)
elif period == "Pra-COVID":
    if 2019 in data_1['tahun'].unique():
        base_data_from_1 = data_1[data_1['tahun'] == 2019].copy()
        tahun_filter_value = 2019
        tahun_display = "2019"
    else:
        base_data_from_1 = data_1[data_1['tahun'] == data_1['tahun'].min()].copy()
        tahun_filter_value = data_1['tahun'].min()
        tahun_display = str(tahun_filter_value)
else:
    base_data_from_1 = data_1.copy()

data_2_categories = data_2[['nama_kabupaten_kota', period]].copy()
data_2_categories.rename(columns={period: 'Kategori_Periode_Specific'}, inplace=True)

filtered_data = pd.merge(
    base_data_from_1,
    data_2_categories,
    on='nama_kabupaten_kota',
    how='left'
)

filtered_data['Kategori'] = filtered_data['Kategori_Periode_Specific']
filtered_data.drop(columns=['Kategori_Periode_Specific'], inplace=True)

filtered_data['latitude'] = pd.to_numeric(filtered_data['latitude'], errors='coerce')
filtered_data['longitude'] = pd.to_numeric(filtered_data['longitude'], errors='coerce')

m = folium.Map(location=[-6.2088, 106.8456], zoom_start=5)
cluster_colors = {0: 'red', 1: 'yellow', 2: 'green'}
marker_cluster = MarkerCluster().add_to(m)

for index, row in filtered_data.iterrows():
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        cluster = row['cluster_fix']
        lat = row['latitude']
        lon = row['longitude']
        popup = f"{row['nama_kabupaten_kota']} - {row['Kategori']}"

        folium.CircleMarker(
            location=[lat, lon],
            radius=8,
            color=cluster_colors.get(cluster, 'blue'),
            fill=True,
            fill_color=cluster_colors.get(cluster, 'blue'),
            fill_opacity=0.6,
            popup=popup
        ).add_to(marker_cluster)

st.title(f"Distribusi Cluster - {period}")
st.write("Peta ini menunjukkan distribusi cluster kerentanan ekonomi di setiap kabupaten/kota.")
st.write("""
🔴: Kategori rendah,
🟡: Kategori sedang,
🟢: Kategori tinggi
""")
folium_static(m)

st.write(f"Data untuk Periode {period} {tahun_display if tahun_display else ''};")
st.dataframe(filtered_data[['nama_kabupaten_kota', 'cluster', 'cluster_fix', 'Kategori']])

Overwriting app.py


In [ ]:
!streamlit run app.py



2026-07-30 21:37:55.852 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.21.153.0:8501

